# Assignment 1: Weather Agent (Gemini SDK)

An AI agent built with the **Gemini SDK** that retrieves the current weather for
three locations using **OpenWeather**, calling the weather API **sequentially**,
then computing the **average temperature** in Python.

## How the workflow fits together

```
┌────────────────────────────────────────────────────────────────┐
│  User prompt: "What is the current weather in Kathmandu,       │
│                Delhi, London?"                                 │
└──────────────────────────────┬─────────────────────────────────┘
                               ▼
┌────────────────────────────────────────────────────────────────┐
│  Gemini (gemini-flash-lite-latest) with the tool               │
│  `get_current_weather` attached                                │
│                                                                │
│  Because automatic function calling (AFC) is DISABLED, Gemini  │
│  does not run the tool itself — it only RETURNS a function     │
│  call, and our loop executes it. This makes every call visible │
│  and forces one-call-at-a-time (sequential) behaviour.         │
└──────────────────────────────┬─────────────────────────────────┘
                               ▼
┌────────────────────────────────────────────────────────────────┐
│  Manual tool loop (up to MAX_TOOL_ROUNDS = 5 rounds)           │
│                                                                │
│  round 1 → Gemini asks: get_current_weather("Kathmandu")       │
│            we call OpenWeather, append the result to contents  │
│  round 2 → Gemini asks: get_current_weather("Delhi")           │
│  round 3 → Gemini asks: get_current_weather("London")          │
│  round 4 → Gemini sees all three results and writes a summary  │
└──────────────────────────────┬─────────────────────────────────┘
                               ▼
┌────────────────────────────────────────────────────────────────┐
│  Python computes the average from the collected tool results   │
│  (never delegated to the LLM — models are bad at arithmetic).  │
└────────────────────────────────────────────────────────────────┘
```

**Key design decisions**

| Decision | Why |
|---|---|
| Manual function calling (AFC disabled) | Each tool call is visible and strictly sequential, as the assignment requires |
| Average computed in Python | Never trust an LLM with arithmetic |
| Shared config in `constants.py` | Prompts, locations and messages live in one editable place |
| Round cap (`MAX_TOOL_ROUNDS`) | Safety net so a runaway loop can't spin forever |

## What you'll see when you run it

A normal successful run prints, roughly:

```
[AGENT] Requested tool: get_current_weather with args {'city': 'Kathmandu'}
[TOOL] Fetching current weather for 'Kathmandu'...
[AGENT] Requested tool: get_current_weather with args {'city': 'Delhi'}
[TOOL] Fetching current weather for 'Delhi'...
[AGENT] Requested tool: get_current_weather with args {'city': 'London'}
[TOOL] Fetching current weather for 'London'...

--- Agent summary ---
Kathmandu is 28°C, Delhi 33°C and London 17°C ...

--- Collected temperatures ---
Kathmandu: 28.1°C (scattered clouds)
Delhi: 33.4°C (haze)
London: 17.2°C (light rain)

Average temperature across 3 locations: 26.23°C
```


## 1. Setup

Install the dependencies once (notebook equivalents of the README):

```bash
python -m pip install google-genai python-dotenv requests
```

Then create a `.env` file next to this notebook (copy `.env.example`) with:

- `GEMINI_API_KEY` — from Google AI Studio
- `OPENWEATHER_API_KEY` — from openweathermap.org

> Note: a fresh OpenWeather key takes ~10 minutes to a couple of hours to
> activate — a 401 right after signup is normal.


In [1]:
# Imports and environment check — run this cell first.
import os

from dotenv import load_dotenv
from google import genai

# Load .env from this directory (GEMINI_API_KEY, OPENWEATHER_API_KEY)
load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    raise SystemExit("GEMINI_API_KEY missing — copy .env.example to .env and fill it in.")
if not os.getenv("OPENWEATHER_API_KEY"):
    raise SystemExit("OPENWEATHER_API_KEY missing — copy .env.example to .env and fill it in.")

print("API keys loaded ✔")


SystemExit: GEMINI_API_KEY missing — copy .env.example to .env and fill it in.

/home/ubuntu/Desktop/projects/ai-agent-assignments/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. Shared configuration

All prompts, locations and console messages live in `constants.py` so the
notebook stays readable and the config stays editable in one place.


In [ ]:
# Shared config: model, locations, prompts, console messages.
from constants import (
    LOCATIONS,          # ["Kathmandu", "Delhi", "London"]
    MAX_TOOL_ROUNDS,    # safety cap on the tool-calling loop (5)
    MODEL,              # "gemini-flash-lite-latest"
    MSG_AGENT_REQUEST, MSG_AGENT_SUMMARY, MSG_AVERAGE, MSG_COLLECTED,
    MSG_INCOMPLETE, MSG_NO_CONTENT, MSG_ROUND_CAP, MSG_TEMP_LINE,
    MSG_TOOL_FETCH, PROMPT_TEMPLATE, SYSTEM_INSTRUCTION,
)

print(f"Model: {MODEL}")
print(f"Locations: {LOCATIONS}")


## 3. Sanity check: is the API key working?

Before building the agent, send a trivial prompt to confirm the SDK + key are
functional. (This is the old `chatbot.py` script, kept as a cell.)

**Expected output** (varies per run):

```
--- Sending test prompt to gemini-flash-lite-latest ---
--- Response ---
Hello! I'm working correctly and ready to help you.
```


In [ ]:
# Minimal chatbot round-trip to verify the Gemini API key.
client = genai.Client()

print(f"--- Sending test prompt to {MODEL} ---")
response = client.models.generate_content(
    model=MODEL,
    contents="Hello, world! Reply with one short sentence to confirm you are working.",
)

print("--- Response ---")
print(response.text)


## 4. The tool: `get_current_weather(city)`

A plain Python function that hits the OpenWeather **current-weather** API
(metric units) and returns a small dict. Gemini sees this function's signature
and docstring when we attach it via `tools=[get_current_weather]` — the docstring
is effectively the tool's description, so it matters.

Error handling notes:

- `404` → unknown city, returned as an error dict (the agent can react to it)
- `401` → key invalid or not yet activated (normal right after signup)
- network failures → caught and returned as an error dict instead of crashing


In [ ]:
import requests

from constants import MSG_TOOL_FETCH

def get_current_weather(city: str) -> dict:
    """Returns the current temperature (Celsius) and conditions for a city.

    Args:
        city: The city name to look up (e.g. 'Kathmandu', 'London').
    """
    print(MSG_TOOL_FETCH.format(city=city))
    try:
        resp = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={"q": city, "appid": os.getenv("OPENWEATHER_API_KEY"), "units": "metric"},
            timeout=10,
        )
        if resp.status_code == 404:
            return {"error": f"City '{city}' not found."}
        if resp.status_code == 401:
            return {"error": "Invalid or not-yet-active OpenWeather API key."}
        resp.raise_for_status()
        data = resp.json()
        return {
            "city": data.get("name", city),
            "temperature_c": data["main"]["temp"],
            "conditions": data["weather"][0]["description"],
        }
    except requests.RequestException as exc:
        return {"error": f"Weather request failed for '{city}': {exc}"}

# Try it directly, bypassing the agent — expect something like:
# {'city': 'Kathmandu', 'temperature_c': 28.1, 'conditions': 'scattered clouds'}
get_current_weather("Kathmandu")


## 5. The agent loop: manual (sequential) function calling

This is the heart of the assignment. Each round:

1. Send the conversation (`contents`) to Gemini with the tool attached.
2. If Gemini returns a **function call**, execute it ourselves, append the
   tool result to the conversation, and loop.
3. If Gemini returns plain **text**, it has finished — that's the summary.

Because AFC is disabled, Gemini returns **at most one call per round**, which
is what makes the calls strictly sequential. `MAX_TOOL_ROUNDS` guards the loop.


In [ ]:
from constants import (
    MAX_TOOL_ROUNDS, MSG_AGENT_REQUEST, MSG_AGENT_SUMMARY, MSG_AVERAGE,
    MSG_COLLECTED, MSG_INCOMPLETE, MSG_NO_CONTENT, MSG_ROUND_CAP,
    MSG_TEMP_LINE, PROMPT_TEMPLATE, SYSTEM_INSTRUCTION,
)
from google.genai import types

available_tools = {"get_current_weather": get_current_weather}

def run_weather_agent(cities: list[str]) -> None:
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        tools=[get_current_weather],
        temperature=0.0,
        # AFC disabled => Gemini RETURNS the call, we execute it. Sequential.
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )

    prompt = PROMPT_TEMPLATE.format(cities=", ".join(cities))
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]

    weather_results: list[dict] = []

    for round_no in range(1, MAX_TOOL_ROUNDS + 1):
        response = client.models.generate_content(
            model=MODEL, contents=contents, config=config
        )
        candidate = response.candidates[0] if response.candidates else None
        if candidate is None or candidate.content is None:
            print(MSG_NO_CONTENT.format(round_no=round_no))
            break
        contents.append(candidate.content)
        function_calls = response.function_calls
        if not function_calls:
            # No more tool calls => the model wrote its final summary.
            print(MSG_AGENT_SUMMARY)
            print(response.text)
            break

        for call in function_calls:
            fn_name = call.name or ""
            fn_args = dict(call.args) if call.args else {}
            print(MSG_AGENT_REQUEST.format(fn_name=fn_name, fn_args=fn_args))
            fn = available_tools.get(fn_name)
            result = fn(**fn_args) if fn else {"error": f"Unknown tool '{fn_name}'"}
            if isinstance(result, dict) and "temperature_c" in result:
                weather_results.append(result)
            contents.append(
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_function_response(
                            name=fn_name, response=result
                        )
                    ],
                )
            )
    else:
        print(MSG_ROUND_CAP)

    # Average computed in Python — never trust the LLM with arithmetic.
    print(MSG_COLLECTED)
    temps = [w["temperature_c"] for w in weather_results]
    for w in weather_results:
        print(MSG_TEMP_LINE.format(city=w["city"], temp=w["temperature_c"], conditions=w["conditions"]))
    if len(temps) == len(cities) and temps:
        avg = sum(temps) / len(temps)
        print(MSG_AVERAGE.format(count=len(temps), avg=f"{avg:.2f}"))
    else:
        print(MSG_INCOMPLETE.format(got=len(temps), expected=len(cities)))


## 6. Run the agent

Sends the prompt for all three locations and drives the loop. Watch the
`[AGENT]` / `[TOOL]` lines to see the sequential tool calls happening.

**Expected output** (temperatures vary with live weather):

```
[AGENT] Requested tool: get_current_weather with args {'city': 'Kathmandu'}
[TOOL] Fetching current weather for 'Kathmandu'...
[AGENT] Requested tool: get_current_weather with args {'city': 'Delhi'}
[TOOL] Fetching current weather for 'Delhi'...
[AGENT] Requested tool: get_current_weather with args {'city': 'London'}
[TOOL] Fetching current weather for 'London'...

--- Agent summary ---
Kathmandu is 28°C with scattered clouds, Delhi is 33°C with haze, and
London is 17°C with light rain.

--- Collected temperatures ---
Kathmandu: 28.1°C (scattered clouds)
Delhi: 33.4°C (haze)
London: 17.2°C (light rain)

Average temperature across 3 locations: 26.23°C
```

If a location fails (e.g. 401 while the OpenWeather key is still activating)
you'll see the error dict in the tool output and the run ends with
`Could not compute average: got 2/3 results.`


In [ ]:
run_weather_agent(LOCATIONS)


## 7. Recap

1. **Config** — model, cities and prompts come from `constants.py`.
2. **Tool** — `get_current_weather` wraps OpenWeather; its docstring is what Gemini sees.
3. **Loop** — manual function calling: Gemini proposes a call, we execute it, feed the result back, repeat until the model answers in text.
4. **Average** — computed in Python from the collected results.

Try changing things in `constants.py`: edit `LOCATIONS` to run other cities, or
raise `MAX_TOOL_ROUNDS` if you add more locations (you need roughly one round
per city plus one final round).
